# Vendor Performance — Extended ML Models

Four additional models that complement `ML_Models.ipynb`:

| # | Model | Type | Why it's different from the originals |
|---|-------|------|---------------------------------------|
| 4 | K-Means Vendor Clustering | Unsupervised | Finds natural archetypes with no predefined labels |
| 5 | Gross Profit Dollar Prediction | Regression | Predicts absolute $ value, not just margin % |
| 6 | Vendor Scorecard | Rule + ML hybrid | Transparent 0–100 score for each vendor |
| 7 | SHAP Explainability | Interpretability | Why did the model make *this* prediction? |

## 0. Imports & Data Setup

Same cleaning pipeline as `ML_Models.ipynb` — run this before any model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import shap
import warnings

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

In [ ]:
# ── Load & clean (same logic as ML_Models.ipynb) ────────────────────────────
df = pd.read_csv('dataset/vendor_sales_summary.csv', sep=';')
df = df[df['TotalSalesDollars'] > 0].copy()
df = df[df['ProfitMargin'] > -200].copy()

st_cap = df['StockTurnover'].quantile(0.99)
df['StockTurnover'] = df['StockTurnover'].clip(upper=st_cap)

order_map = {'Small': 0, 'Medium': 1, 'Large': 2, 'X-Large': 3}
df['OrderSize'] = pd.qcut(
    df['TotalPurchaseQuantity'], q=4,
    labels=['Small', 'Medium', 'Large', 'X-Large']
).map(order_map)

df['PriceMarkup']    = (df['ActualPrice'] - df['PurchasePrice']) / df['PurchasePrice']
df['FreightPerUnit'] = df['TotalFreightCost'] / df['TotalPurchaseQuantity'].clip(lower=1)
df['TaxPerUnit']     = df['TotalExciseTax']   / df['TotalSalesQuantity'].clip(lower=1)

print(f"Dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## Model 4 — K-Means Vendor Clustering

> **Goal:** Let the data group itself, without any predefined High/Medium/Low labels.
> This reveals natural vendor archetypes — useful for tailored strategies (e.g. *push volume* vs *protect margin* vs *review pricing*).

### Step 1 — Find the right number of clusters (Elbow + Silhouette)

In [ ]:
CLUSTER_FEATURES = [
    'ProfitMargin', 'StockTurnover',
    'TotalSalesDollars', 'TotalPurchaseDollars',
    'SalestoPurchaseRatio', 'PriceMarkup',
    'FreightPerUnit', 'GrossProfit'
]

X_cluster = df[CLUSTER_FEATURES].copy()

scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X_cluster)

k_range = range(2, 9)
inertias   = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=3000, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, marker='o', color='steelblue')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — choose k where curve bends')

axes[1].plot(list(k_range), silhouettes, marker='s', color='darkorange')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Score — choose k with highest value')

plt.tight_layout()
plt.show()

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f"Best k by silhouette: {best_k}  (score = {max(silhouettes):.4f})")

### Step 2 — Fit K-Means and profile the clusters

In [ ]:
# Use best_k found above (or override manually: best_k = 4)
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# Profile: median of key metrics per cluster
cluster_profile = df.groupby('Cluster')[CLUSTER_FEATURES].median().round(2)
cluster_profile['Count'] = df['Cluster'].value_counts().sort_index()

print("Cluster Profiles (median values):")
cluster_profile

In [ ]:
# ── Radar / Spider chart — one spoke per feature, one line per cluster ────────
from matplotlib.patches import FancyArrowPatch

# Normalise cluster centres back to 0-1 for radar
centres = pd.DataFrame(
    scaler_cluster.inverse_transform(km_final.cluster_centers_),
    columns=CLUSTER_FEATURES
)

radar_features = ['ProfitMargin', 'StockTurnover', 'TotalSalesDollars',
                  'SalestoPurchaseRatio', 'GrossProfit', 'PriceMarkup']

norm = MinMaxScaler()
centres_norm = pd.DataFrame(
    norm.fit_transform(centres[radar_features]),
    columns=radar_features
)

N = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors_radar = plt.cm.Set2(np.linspace(0, 1, best_k))

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for i, row in centres_norm.iterrows():
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, color=colors_radar[i], linewidth=2, label=f'Cluster {i}')
    ax.fill(angles, vals, color=colors_radar[i], alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=9)
ax.set_title('Cluster Radar — normalised feature centroids', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
# ── 2D scatter — ProfitMargin vs TotalSalesDollars coloured by cluster ────────
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df['TotalSalesDollars'], df['ProfitMargin'],
    c=df['Cluster'], cmap='Set2',
    alpha=0.45, s=18, edgecolors='none'
)
plt.colorbar(scatter, label='Cluster')
plt.xscale('log')
plt.xlabel('Total Sales Dollars (log scale)')
plt.ylabel('Profit Margin (%)')
plt.title('Vendor Clusters — Profit Margin vs Sales Volume')
plt.tight_layout()
plt.show()

In [ ]:
# ── Name the clusters based on the profile ────────────────────────────────────
# Adjust these labels after reviewing the radar + profile table above
# (the right archetype name depends on which k the silhouette chose)

# Example for k=4 — edit as needed:
# cluster_names = {
#     0: 'High Volume / Thin Margin',
#     1: 'Niche / High Margin',
#     2: 'Low Activity / At Risk',
#     3: 'Balanced Performers'
# }
# df['ClusterName'] = df['Cluster'].map(cluster_names)

print("Review the radar chart and profile table above, then fill in cluster_names.")
print("Uncomment the block above and add your own archetype labels.")

---
## Model 5 — Gross Profit Dollar Prediction (Regression)

> **Goal:** Predict the *absolute dollar gross profit* a vendor-brand will generate.
> This is intentionally different from Model 1 (% margin): a vendor with 40% margin
> on \$500 total sales is far less valuable than one with 20% margin on \$500K.

**Leakage note:** `ProfitMargin`, `TotalSalesDollars`, `TotalSalesPrice`, and `TotalPurchaseDollars`
are strong predictors but would give away the answer in a real prediction scenario
(you wouldn't know these for a new vendor). They are kept here since this is a
batch scoring task on historical data — flag them if you apply the model to new vendors.

In [ ]:
TARGET_GP = 'GrossProfit'

# Log-transform target: GrossProfit is very right-skewed.
# We predict log(GrossProfit + 1) and invert after.
y_gp = np.log1p(df[TARGET_GP].clip(lower=0))

GP_FEATURES = [
    'PurchasePrice', 'Volume', 'ActualPrice',
    'TotalPurchaseQuantity', 'TotalSalesQuantity',
    'TotalExciseTax', 'TotalFreightCost',
    'SalestoPurchaseRatio', 'StockTurnover',
    'PriceMarkup', 'FreightPerUnit', 'TaxPerUnit',
    'OrderSize'
]

X_gp = df[GP_FEATURES].copy()

X_train_gp, X_test_gp, y_train_gp, y_test_gp = train_test_split(
    X_gp, y_gp, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train_gp.shape} | Test: {X_test_gp.shape}")
print(f"Target (log scale) — mean: {y_gp.mean():.2f}, std: {y_gp.std():.2f}")

In [ ]:
# ── XGBoost (log-transformed target) ────────────────────────────────────────
xgb_gp = XGBRegressor(
    n_estimators=400, learning_rate=0.04, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, verbosity=0
)
xgb_gp.fit(X_train_gp, y_train_gp)

# Invert log transform for evaluation
y_pred_gp_log = xgb_gp.predict(X_test_gp)
y_pred_gp     = np.expm1(y_pred_gp_log)
y_true_gp     = np.expm1(y_test_gp)

mae_gp  = mean_absolute_error(y_true_gp, y_pred_gp)
rmse_gp = np.sqrt(mean_squared_error(y_true_gp, y_pred_gp))
r2_gp   = r2_score(y_true_gp, y_pred_gp)

print(f"Gross Profit Prediction — XGBoost (log-transformed target)")
print(f"  MAE  : ${mae_gp:,.2f}")
print(f"  RMSE : ${rmse_gp:,.2f}")
print(f"  R²   : {r2_gp:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted (log scale for readability)
ax = axes[0]
ax.scatter(y_true_gp, y_pred_gp, alpha=0.3, s=15, color='teal')
lims = [0, y_true_gp.quantile(0.98)]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual Gross Profit ($)')
ax.set_ylabel('Predicted Gross Profit ($)')
ax.set_title('Actual vs Predicted — Gross Profit')
ax.legend()

# Residuals
ax2 = axes[1]
residuals = y_true_gp - y_pred_gp
ax2.scatter(y_pred_gp, residuals, alpha=0.3, s=15, color='coral')
ax2.axhline(0, color='black', linestyle='--', lw=1)
ax2.set_xlabel('Predicted Gross Profit ($)')
ax2.set_ylabel('Residual ($)')
ax2.set_title('Residual Plot')
ax2.set_xlim(left=0, right=y_pred_gp.quantile(0.97))
ax2.set_ylim(y_true_gp.quantile(0.02), y_true_gp.quantile(0.97))

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feat_imp_gp = pd.Series(
    xgb_gp.feature_importances_,
    index=X_gp.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(x=feat_imp_gp.values, y=feat_imp_gp.index, palette='crest')
plt.title('XGBoost — Feature Importances (Gross Profit Prediction)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## Model 6 — Vendor Scorecard (Composite 0–100 Score)

> **Goal:** Give every vendor-brand a single, transparent performance score between 0 and 100,
> with a per-dimension breakdown. Unlike the classifier, this is fully explainable to stakeholders
> — no black box involved.

### Scoring Dimensions & Weights

| Dimension | Feature used | Weight |
|-----------|-------------|--------|
| Profitability | `ProfitMargin` | 30% |
| Sales Volume | `TotalSalesDollars` | 25% |
| Inventory Efficiency | `StockTurnover` | 20% |
| Pricing Power | `SalestoPurchaseRatio` | 15% |
| Absolute Profit | `GrossProfit` | 10% |

Each dimension is percentile-ranked within the dataset (0–100), then weighted and summed.

In [ ]:
SCORE_DIMS = {
    'ProfitMargin':       0.30,
    'TotalSalesDollars':  0.25,
    'StockTurnover':      0.20,
    'SalestoPurchaseRatio': 0.15,
    'GrossProfit':        0.10,
}

# Percentile rank each dimension across all vendors (0 = worst, 100 = best)
for col in SCORE_DIMS:
    df[f'Score_{col}'] = df[col].rank(pct=True) * 100

# Weighted composite score
df['VendorScore'] = sum(
    df[f'Score_{col}'] * weight
    for col, weight in SCORE_DIMS.items()
)

# Tier from score
df['ScoreTier'] = pd.cut(
    df['VendorScore'],
    bins=[0, 33, 67, 100],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

print(df['ScoreTier'].value_counts())
print()
print(f"Score range: {df['VendorScore'].min():.1f} – {df['VendorScore'].max():.1f}")
print(f"Score mean : {df['VendorScore'].mean():.1f}")

In [ ]:
# Score distribution by tier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tier_colors = {'High': '#2ecc71', 'Medium': '#f39c12', 'Low': '#e74c3c'}

# Histogram of vendor scores
for tier in ['Low', 'Medium', 'High']:
    subset = df[df['ScoreTier'] == tier]['VendorScore']
    axes[0].hist(subset, bins=30, alpha=0.6, label=tier, color=tier_colors[tier])
axes[0].set_xlabel('Vendor Score (0–100)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Vendor Scores by Tier')
axes[0].legend()

# Stacked bar: mean dimension contribution per tier
dim_cols = [f'Score_{col}' for col in SCORE_DIMS]
nice_names = ['Profitability', 'Sales Volume', 'Inventory', 'Pricing Power', 'Abs. Profit']
dim_by_tier = df.groupby('ScoreTier', observed=True)[dim_cols].mean()

bottom = np.zeros(3)
palette = plt.cm.Pastel1(np.linspace(0, 1, len(dim_cols)))
for i, (col, name) in enumerate(zip(dim_cols, nice_names)):
    axes[1].bar(
        dim_by_tier.index, dim_by_tier[col],
        bottom=bottom, label=name, color=palette[i]
    )
    bottom += dim_by_tier[col].values

axes[1].set_ylabel('Cumulative Percentile Score')
axes[1].set_title('Mean Score Dimension Breakdown by Tier')
axes[1].legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Top 20 and Bottom 20 vendors by composite score ──────────────────────────
report_cols = [
    'VendorName', 'Description',
    'VendorScore', 'ScoreTier',
    'ProfitMargin', 'TotalSalesDollars',
    'StockTurnover', 'GrossProfit'
]

top20 = df.nlargest(20, 'VendorScore')[report_cols].reset_index(drop=True)
bot20 = df.nsmallest(20, 'VendorScore')[report_cols].reset_index(drop=True)

print("\n── TOP 20 VENDORS ──")
display(top20.style
    .background_gradient(subset=['VendorScore'], cmap='Greens')
    .format({'ProfitMargin': '{:.1f}%', 'StockTurnover': '{:.3f}',
             'GrossProfit': '${:,.0f}', 'TotalSalesDollars': '${:,.0f}',
             'VendorScore': '{:.1f}'}))

print("\n── BOTTOM 20 VENDORS ──")
display(bot20.style
    .background_gradient(subset=['VendorScore'], cmap='Reds_r')
    .format({'ProfitMargin': '{:.1f}%', 'StockTurnover': '{:.3f}',
             'GrossProfit': '${:,.0f}', 'TotalSalesDollars': '${:,.0f}',
             'VendorScore': '{:.1f}'}))

In [ ]:
# ── Export scorecard to CSV ──────────────────────────────────────────────────
scorecard_export = df[
    ['VendorNumber', 'VendorName', 'Brand', 'Description', 'VendorScore', 'ScoreTier']
    + [f'Score_{col}' for col in SCORE_DIMS]
].sort_values('VendorScore', ascending=False)

scorecard_export.columns = (
    ['VendorNumber', 'VendorName', 'Brand', 'Description', 'VendorScore', 'ScoreTier']
    + ['Score_Profitability', 'Score_SalesVolume', 'Score_Inventory',
       'Score_PricingPower', 'Score_AbsProfit']
)

scorecard_export.to_csv('dataset/vendor_scorecard.csv', index=False)
print(f"Scorecard saved — {len(scorecard_export):,} vendors")

---
## Model 7 — SHAP Explainability

> **Goal:** Open the XGBoost black boxes from `ML_Models.ipynb` and answer:
> *Why did this vendor get this prediction?*
>
> SHAP (SHapley Additive exPlanations) assigns each feature a contribution value per row,
> so you can audit any individual vendor's score.

We rebuild the two XGBoost models from `ML_Models.ipynb` here (or you can import them
using `joblib` if you saved them).

In [ ]:
# ── Re-build the regression model (Profit Margin) ───────────────────────────
FEATURE_COLS = [
    'PurchasePrice', 'Volume', 'ActualPrice',
    'TotalPurchaseQuantity', 'TotalPurchaseDollars',
    'TotalSalesQuantity',   'TotalSalesDollars',
    'TotalSalesPrice',      'TotalExciseTax',
    'TotalFreightCost',
    'SalestoPurchaseRatio',
    'PriceMarkup', 'FreightPerUnit', 'TaxPerUnit', 'OrderSize'
]

X_reg = df[FEATURE_COLS]
y_reg = df['ProfitMargin']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

xgb_reg_shap = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, random_state=RANDOM_STATE, verbosity=0
)
xgb_reg_shap.fit(X_train_r, y_train_r)
print("Regression model trained.")

In [ ]:
# ── SHAP values for regression model ────────────────────────────────────────
explainer_reg = shap.TreeExplainer(xgb_reg_shap)

# Use test set for SHAP (sample for speed if dataset is large)
X_shap_sample = X_test_r.sample(min(1000, len(X_test_r)), random_state=RANDOM_STATE)
shap_values_reg = explainer_reg.shap_values(X_shap_sample)

print(f"SHAP values computed for {len(X_shap_sample)} test samples.")

In [ ]:
# ── Global: Feature importance (mean |SHAP|) ────────────────────────────────
plt.figure()
shap.summary_plot(
    shap_values_reg, X_shap_sample,
    plot_type='bar',
    show=False
)
plt.title('Mean |SHAP| — Global Feature Importance (Profit Margin Regression)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Beeswarm: direction + magnitude of each feature's effect ────────────────
plt.figure()
shap.summary_plot(shap_values_reg, X_shap_sample, show=False)
plt.title('SHAP Beeswarm — How each feature shifts Profit Margin predictions')
plt.tight_layout()
plt.show()

# Reading guide:
# • Red = high feature value, Blue = low feature value
# • Right of centre = pushed prediction UP, Left = pushed prediction DOWN

In [ ]:
# ── Local: Waterfall for a single vendor ────────────────────────────────────
# Change idx to inspect any specific vendor (row index in the test set)

idx = 0   # ← change to any row index you want to audit

vendor_row = X_shap_sample.iloc[idx]
vendor_name = df.loc[X_shap_sample.index[idx], 'VendorName']
vendor_desc = df.loc[X_shap_sample.index[idx], 'Description']
actual_margin = df.loc[X_shap_sample.index[idx], 'ProfitMargin']

print(f"Explaining prediction for: {vendor_name} — {vendor_desc}")
print(f"Actual ProfitMargin: {actual_margin:.2f}%")

shap_exp = shap.Explanation(
    values     = shap_values_reg[idx],
    base_values= explainer_reg.expected_value,
    data       = vendor_row.values,
    feature_names = X_shap_sample.columns.tolist()
)

plt.figure()
shap.plots.waterfall(shap_exp, show=False)
plt.title(f'SHAP Waterfall — {vendor_name[:40]}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Dependence plot: how one feature drives SHAP values ─────────────────────
# The most impactful feature (by mean |SHAP|) is plotted automatically.
top_feature = pd.Series(
    np.abs(shap_values_reg).mean(axis=0),
    index=X_shap_sample.columns
).idxmax()

print(f"Plotting dependence for top feature: {top_feature}")

plt.figure()
shap.dependence_plot(
    top_feature, shap_values_reg, X_shap_sample,
    show=False
)
plt.title(f'SHAP Dependence — {top_feature} vs its contribution to Profit Margin')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP for the Classifier (Performance Category) ───────────────────────────
from sklearn.preprocessing import MinMaxScaler

scaler_label = MinMaxScaler()
scores = scaler_label.fit_transform(df[['ProfitMargin', 'StockTurnover']])
df['CompositeScore'] = scores[:, 0] + scores[:, 1]
df['PerformanceCategory'] = pd.qcut(
    df['CompositeScore'], q=3, labels=['Low', 'Medium', 'High']
)

le = LabelEncoder()
y_clf = le.fit_transform(df['PerformanceCategory'])
X_clf = df[FEATURE_COLS]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=RANDOM_STATE
)

xgb_clf_shap = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=RANDOM_STATE, verbosity=0
)
xgb_clf_shap.fit(X_train_c, y_train_c)
print("Classifier model trained.")

In [ ]:
explainer_clf = shap.TreeExplainer(xgb_clf_shap)

X_shap_clf = X_test_c.sample(min(500, len(X_test_c)), random_state=RANDOM_STATE)
shap_values_clf = explainer_clf.shap_values(X_shap_clf)  # shape: [n_classes, n_rows, n_features]

class_names = le.classes_  # ['High', 'Low', 'Medium'] or alphabetical

# Beeswarm for each class
fig, axes = plt.subplots(1, len(class_names), figsize=(16, 5))

for i, (cls, ax) in enumerate(zip(class_names, axes)):
    plt.sca(ax)
    shap.summary_plot(
        shap_values_clf[i], X_shap_clf,
        max_display=8, show=False, plot_size=None
    )
    ax.set_title(f'SHAP — class: {cls}', fontsize=11)

plt.suptitle('Feature impact per Performance Class (Classification)', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Model | What it adds over the original 3 |
|-------|-----------------------------------|
| K-Means Clustering | Reveals natural archetypes the rule-based labels miss — different vendors need different strategies |
| Gross Profit $ Prediction | Prioritises vendors by absolute dollar impact, not just % margin |
| Vendor Scorecard | Fully transparent 0–100 score, exportable for stakeholder reports without needing to explain a model |
| SHAP Explainability | Makes the XGBoost models auditable — any prediction can be traced back to specific features for that vendor |

**Recommended order to present to stakeholders:**
1. Scorecard (intuitive, no ML jargon)
2. Clusters (visual, strategy-ready)
3. Classification with SHAP (auditable decisions)
4. Regression models (forecasting / planning)